# 1. Overview

The goal of this notebook is to forecast store sales using the Kaggle Store Sales – Time Series Forecasting dataset.

We will first combine the different files:

train.csv → main sales data (target: sales)

stores.csv → store information (location, type, cluster)

holidays_events.csv → holidays and special events

oil.csv → daily oil prices

transactions.csv → number of transactions per store

After merging, we will perform some feature engineering, such as extracting date parts (day, month, weekday), adding holiday indicators, and encoding store and product families.

Then, we will build Linear Regression model using

Scikit-learn Linear Regression – quick and reliable baseline.

In [ ]:
 
import numpy as np 
import pandas as pd  
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score


# 2. Preprocessing

## 2.1 Merge Data

In [ ]:
 

# Load all datasets
train = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/train.csv", parse_dates=['date'])
stores = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/stores.csv")
holidays = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv", parse_dates=['date'])
oil = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/oil.csv", parse_dates=['date'])
transactions = pd.read_csv("/kaggle/input/store-sales-time-series-forecasting/transactions.csv", parse_dates=['date'])

# Merge store info
df = train.merge(stores, on="store_nbr", how="left")

# Merge transactions
df = df.merge(transactions, on=["date", "store_nbr"], how="left")

# Merge oil prices
df = df.merge(oil, on="date", how="left")

# Merge holiday info
df = df.merge(holidays, on="date", how="left")


# Preview
print(df.shape)

df.describe()

In [ ]:
df.isnull().sum()

## 2.2 Handling Missing Values

In [ ]:
for col in df.select_dtypes(include=np.number).columns: # for numrical columns
    df[col]=df[col].fillna(df[col].median())

for col in df.select_dtypes(include="object").columns:  # for catogrical columns
    df[col]=df[col].fillna(df[col].mode()[0])
df.isnull().sum()

## 2.2 Handling Outliers using IQR

In [ ]:
def remove_outliers_IQR(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] >= lower) & (df[column] <= upper)]
for col in df.select_dtypes(include=np.number).columns:
   df= remove_outliers_IQR(df,col)

# 3. Feature Engineering

In [ ]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["dayofweek"] = df["date"].dt.dayofweek  # 0=Monday, 6=Sunday
df["is_weekend"] = df["dayofweek"].isin([5,6]).astype(int)
df["is_holiday"] = df["type_y"].notnull().astype(int)


In [ ]:
df.info()

# 4. Features Encoding & Scaling

In [ ]:
numric_features=["onpromotion","transactions","dcoilwtico","year","month","day","dayofweek","is_weekend","is_holiday"]
categorical_features=["family","city","state","type_x","type_y","locale","locale_name","description"]

numric_transformer=StandardScaler()
categorical_transformer=OneHotEncoder(handle_unknown="ignore")

preprocessor=ColumnTransformer(
    transformers=[("num",numric_transformer,numric_features),
                  ("cat",categorical_transformer,categorical_features)
                 ]
)

# 5. Train & Test Spilit

In [ ]:
X = df.drop(columns=["sales", "date"])   
y = df["sales"]

  
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# 6. Model Pipelining

In [ ]:

model= Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("regressor",LinearRegression())
]              
  )

In [ ]:

model.fit(X_train, y_train)

# 7. Model Predection

In [ ]:

y_predict=model.predict(X_test)

In [ ]:
lin_reg = model.named_steps["regressor"]

# Coefficients (weights) and intercept
weights = lin_reg.coef_
bias = lin_reg.intercept_

print("Bias (intercept):", bias)
print("Weights shape:", weights.shape)


# 8. Model Evaluation

## 8.1 Get the name of top 10 features impacts

In [ ]:
# Get feature names after preprocessing
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

# Combine with weights
coef_df = pd.DataFrame({
    "feature": feature_names,
    "weight":  weights
})

print(coef_df.sort_values(by="weight", ascending=False).head(10))


## 8.2 Model Score

In [ ]:
rmse=np.sqrt(mean_squared_error(y_predict,y_test))
print(rmse)
r2=r2_score(y_test,y_predict)
print(r2)


## 8.3 Visualizing the diffrence between Predicted y and Actual y

In [ ]:
plt.scatter(y_test, y_predict, alpha=0.6, color="blue")
plt.xlabel(" (y)")
plt.ylabel(" (ŷ)")
plt.title("Linear Regression with Preprocessing + Gradient Descent")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         color="red", linestyle="--")
plt.show()

In [ ]:
import joblib

joblib.dump(model,"store_sales_forcasting_using_LG.pkl")

load_model=joblib.load("store_sales_forcasting_using_LG.pkl")

sample_predict=load_model.predict(X_test[:5])
sample_predict

# Conclusion – Linear Regression

The Linear Regression model provided a basic baseline, achieving RMSE of ~61 and R² of ~0.53.

While it captures some relationships in the data, it underfits, especially for higher sales values, due to its linear nature.

The model is useful for interpretability and benchmarking, but not sufficient for high-accuracy forecasting.

To improve performance, applying log-transformation of sales, adding polynomial or interaction terms, or using advanced models (Ridge/Lasso regression, XGBoost, LightGBM) is recommended.